
# <font color="green">Passing many parameters</font>

## Problem

* Write a function `call_many_args(base)` __in assembly__ that calls the (given) twenty-parameter function `add_many` with the arguments `base, base+1, ..., base+19`, and returns its result.
* That is, translate the following C function into assembly:
```
long add_many(long, long, long, long, long, long, long, long, long, long,
              long, long, long, long, long, long, long, long, long, long); /* given */

long call_many_args(long base) {
  return add_many(base, base+1, base+2, base+3, base+4, base+5, base+6, base+7,
                  base+8, base+9, base+10, base+11, base+12, base+13, base+14,
                  base+15, base+16, base+17, base+18, base+19);
}
```
* The first eight arguments go in `x0`–`x7`. The remaining twelve must be written to the stack at `[sp]`, `[sp, 8]`, `[sp, 16]`, ... **before** the `bl add_many`. Remember to reserve that stack space (keeping `sp` 16-byte aligned) and to set up a frame (save `x29`/`x30`) because you make a call.
* Fill in the skeleton `call_many_args.s` (after `// ------- write your answer here -------`).
* The checker `check_call_many_args.c` provides `add_many` and verifies your result. If you see `OK`s and no errors, you are done.

## Hints

* This is the caller's side of the previous problem (*Receiving many parameters*). The same ABI rule applies: the first eight integer arguments are passed in `x0`–`x7`, and the rest on the stack.
* To pass arguments on the stack, the caller reserves space below `sp` and stores the extra arguments at `[sp]`, `[sp, 8]`, ... (the ninth argument at the lowest address). The callee then reads them from those same `sp`-relative locations.
* The *Observe* cells below contain a simple example, `call_add10`, which calls a 10-parameter function with constants (so only the 9th and 10th arguments go on the stack). Compile it and observe how the caller reserves stack space and stores those arguments before the `bl` --- then do the same for twenty arguments, with the values `base, base+1, ...` computed from your argument.



# 1. AI Tutor
## 1-1. Prepare
* Your personal AI tutor is provided for questions and feedback.
* Execute the following cell before you use it.

In [ ]:
import heytutor

## 1-2. Examples
* A general question
```
%%hey
What does the `ldr` instruction do in ARM64?
```

* A hint on this specific problem
```
%%hey problem_file=call_many_args.md
Give me a hint on this problem.

{problem}
```

* Builtin variables usable in `%%hey` cells
  * `{file:FILENAME}` is the content of FILE
  * `{bash[-1]}` is the output of the last `%%bash_` cell, `{bash[-2]}` the second last, etc.
  * `{problem}` is the content of the file you specify by `%%hey problem_file=foo.md`
  * `{answer}` is the content of the file you specify by `%%hey answer_file=foo.s`


# 2. Observe: compile example functions
* Before writing your own assembly, it helps to see what the compiler generates for related example functions.
* Running the first cell below writes `explore.c` (some small example functions related to this problem).
* The second cell compiles it with `gcc -O3 -S` and prints the generated assembly.
* Feel free to edit `explore.c` (change the code, add functions, change constants) and re-run the two cells to see how the assembly changes.

In [ ]:
%%writefile_ explore.c
/* Calling a function with more than eight arguments: the first eight go in
   x0..x7, and the rest must be written to the STACK at [sp], [sp,#8], ...
   before the 'bl'. This simple example calls a 10-parameter function, so only
   the 9th and 10th arguments are passed on the stack. Observe how the caller
   reserves stack space and stores those two before the call. */
long add10(long, long, long, long, long, long, long, long, long, long);
long call_add10(void) {
  return add10(1, 2, 3, 4, 5, 6, 7, 8, 9, 10);
}

In [ ]:
%%bash_
gcc -O3 -S explore.c
cat explore.s


# 3. Your Answer (assembly)
* Running the cell below writes the skeleton assembly file `call_many_args.s`.
* Fill in your instructions after the line `// ------- write your answer here -------`, then run the cell again to save it.

In [ ]:
%%writefile_ call_many_args.s
	.arch armv8-a
	.file	"call_many_args.c"
	.text
	.align	2
	.p2align 4,,11
	.global	call_many_args
	.type	call_many_args, %function
call_many_args:
.LFB0:
	.cfi_startproc
	// ------- write your answer here -------
	.cfi_endproc
.LFE0:
	.size	call_many_args, .-call_many_args
	.ident	"GCC: (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0"
	.section	.note.GNU-stack,"",@progbits


# 4. Checker
* The following C program calls your `call_many_args` function and checks the result against a reference computed in C.

In [ ]:
%%writefile_ check_call_many_args.c
#include <assert.h>
#include <stdio.h>
#include <stdlib.h>
long call_many_args(long base);

/* the function your code must call (sum of its 20 arguments) */
long add_many(long a00, long a01, long a02, long a03, long a04, long a05, long a06, long a07, long a08, long a09, long a10, long a11, long a12, long a13, long a14, long a15, long a16, long a17, long a18, long a19) {
  return a00 + a01 + a02 + a03 + a04 + a05 + a06 + a07 + a08 + a09 + a10 + a11 + a12 + a13 + a14 + a15 + a16 + a17 + a18 + a19;
}

int main(int argc, char ** argv) {
  assert(argc == 2);
  long base = atol(argv[1]);
  long r = call_many_args(base);
  long rc = 0;
  for (int i = 0; i < 20; i++) rc += base + i;   /* add_many(base, base+1, ..., base+19) */
  if (r == rc) { printf("OK %ld %ld\n", r, rc); return 0; }
  else { printf("NG %ld %ld\n", r, rc); return 1; }
}


# 5. Compile
* Compile your assembly together with the checker.
* If you get an error, fix `call_many_args.s` above and recompile.

In [ ]:
%%bash_
gcc -o check_call_many_args -O3 check_call_many_args.c call_many_args.s -lm


# 6. Run
* If you see `OK`s and no errors, you are done.

In [ ]:
%%bash_
./check_call_many_args 0
./check_call_many_args 100
./check_call_many_args -10


# 7. If things do not go well
* If your program compiles but does not produce the correct answer, run it within a debugger (gdb).
* Compile with `-O0 -g` first:
```
gcc -o check_call_many_args -O0 -g check_call_many_args.c call_many_args.s -lm
```
* Then, in a terminal (SSH or Jupyter terminal):
```
gdb check_call_many_args
(gdb) break call_many_args
(gdb) run ...        # give the same arguments as above
```
* Step through one instruction at a time with `step`, and inspect registers with `print $x0` or `info registers`.

# 8. Ask Questions or Get Feedback
* You are encouraged to ask for feedback once you think you are done, to know if there is a better answer.

In [ ]:
%%hey problem_file=call_many_args.md answer_file=call_many_args.s

Problem:
{problem}

My Answer:
{answer}

Give me a feedback to my answer.